# 01 — Data Cleaning

Objective: inspect the raw Telco churn dataset thoroughly, fix genuine data-quality issues, and produce a cleaned dataset in `data/processed/`. The raw file under `data/raw/` is never modified.

All reusable cleaning logic lives in `src/data/clean.py`; this notebook applies it, reports the evidence, and documents every decision.

In [1]:
import sys
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(p for p in _candidates if (p / "src").exists())
sys.path.insert(0, str(PROJECT_ROOT))
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

from src.data.clean import (
    profile_dataset, clean_telco, validate_cleaned, save_processed, load_raw,
)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)


In [2]:
df = load_raw()
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Dataset profile

Shape, dtypes, missing values, blank strings, whitespace, duplicates, categorical domains, numeric ranges.

In [3]:
profile = profile_dataset(df)
print('shape:', profile['shape'])
print('\ndtypes:')
for col, dtype in profile['dtypes'].items():
    print(f'  {col:<18} {dtype}')

shape: (7043, 21)

dtypes:
  customerID         str
  gender             str
  SeniorCitizen      int64
  Partner            str
  Dependents         str
  tenure             int64
  PhoneService       str
  MultipleLines      str
  InternetService    str
  OnlineSecurity     str
  OnlineBackup       str
  DeviceProtection   str
  TechSupport        str
  StreamingTV        str
  StreamingMovies    str
  Contract           str
  PaperlessBilling   str
  PaymentMethod      str
  MonthlyCharges     float64
  TotalCharges       str
  Churn              str


In [4]:
print('missing values (excluding TotalCharges blanks):')
missing = {k: v for k, v in profile['missing_by_column'].items() if v > 0}
print(missing if missing else '  none')
print('\nduplicate rows:', profile['duplicate_rows'])
print('duplicate customerIDs:', profile['duplicate_customer_ids'])
print('\nblank strings per column:')
print({k: v for k, v in profile['blank_string_counts'].items() if v > 0})
print('\ncolumns with leading/trailing whitespace:')
print({k: v for k, v in profile['whitespace_hits'].items() if v > 0} or '  none')

missing values (excluding TotalCharges blanks):
  none

duplicate rows: 0
duplicate customerIDs: 0

blank strings per column:
{'TotalCharges': 11}

columns with leading/trailing whitespace:
{'TotalCharges': 11}


In [5]:
print('unexpected categorical values:', profile['unexpected_category_values'] or 'none')
print('\nTotalCharges non-numeric values:', profile['total_charges_non_numeric'])
print('  tenure of those rows:', profile['total_charges_non_numeric_tenure'])
print('  Churn of those rows:', profile['total_charges_non_numeric_churn'])

unexpected categorical values: none

TotalCharges non-numeric values: 11
  tenure of those rows: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Churn of those rows: ['No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No']


**Finding:** the only genuine quality issue is `TotalCharges` — stored as text, with 11 blank strings. No duplicate rows, no duplicate IDs, no missing values elsewhere, no whitespace, and every categorical column matches its expected domain.

## 2. Numeric ranges & outliers

In [6]:
df[['tenure', 'MonthlyCharges']].describe()

,tenure,MonthlyCharges
count,7043.000000,7043.000000
mean,32.371149,64.761692
std,24.559481,30.090047
min,0.000000,18.250000
25%,9.000000,35.500000
50%,29.000000,70.350000
75%,55.000000,89.850000
max,72.000000,118.750000


In [7]:
import numpy as np

tc = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(tc.describe())
print('\nTotalCharges vs MonthlyCharges * tenure consistency (should be close for linear billing):')
ratio = (tc / (df['MonthlyCharges'] * df['tenure'].clip(lower=1))).dropna()
print(ratio.describe()[['min', '25%', '50%', '75%', 'max']])

count    7032.000000
mean     2283.300441
std      2266.771362
min        18.800000
25%       401.450000
50%      1397.475000
75%      3794.737500
max      8684.800000
Name: TotalCharges, dtype: float64

TotalCharges vs MonthlyCharges * tenure consistency (should be close for linear billing):
min    0.689356
25%    0.979546
50%    1.000000
75%    1.019562
max    1.573454
dtype: float64


`tenure` is 0–72 months, `MonthlyCharges` 18.25–118.75 USD, `TotalCharges` 18.8–8684.8 USD — all plausible ranges. `TotalCharges` is not always `MonthlyCharges × tenure` (price changes over time), which is expected. No impossible outliers; extreme values are legitimate long-tenure customers.

## 3. The `TotalCharges` decision

11 blank values, all with `tenure = 0`. These are brand-new customers with no completed billing cycle. Options considered:

| Option | Verdict |
|---|---|
| Drop the 11 rows | Loses real customers; tenure=0 new customers are a legitimate population |
| Impute median/mean | Fabricates billing that never happened; would overstate revenue |
| **Set to 0.0** | Factually correct: no invoice has been issued yet, so total charged to date is genuinely zero |

Chosen: **0.0** — this is a factual value, not a statistical imputation. It is documented everywhere `TotalCharges` is used.

## 4. Apply cleaning

In [8]:
cleaned, log = clean_telco(df)
for key, value in log.items():
    print(f'{key}: {value}')

stripped_columns: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']
duplicate_rows_removed: 0
duplicate_customer_ids: 0
total_charges_blanks: 11
total_charges_blank_tenure_values: [0]
total_charges_blank_all_tenure_zero: True
total_charges_blank_churn_values: ['No']
total_charges_treatment: Blank values set to 0.0: all belong to tenure=0 customers who have not yet been billed, so zero is their true total charged to date (not a statistical imputation).


## 5. Validate & save

In [9]:
checks = validate_cleaned(cleaned)
for name, ok in checks.items():
    print(f'{name}: {ok}')

shape_is_7043x21: True
no_missing_values: True
no_duplicate_customer_ids: True
total_charges_numeric: True
total_charges_non_negative: True
tenure_range_0_72: True
monthly_charges_positive: True
churn_binary: True
no_unexpected_categories: True


In [10]:
out = save_processed(cleaned)
print('saved:', out)
print('final shape:', cleaned.shape)
print('\nfinal dtypes:')
print(cleaned.dtypes)
print('\nfinal missing values:', int(cleaned.isna().sum().sum()))
cleaned.head()

saved: C:\Users\victus\Desktop\pre-wbl\telco-churn-analytics\data\processed\telco_cleaned.csv
final shape: (7043, 21)

final dtypes:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

final missing values: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Cleaning summary

- **Issues found:** `TotalCharges` stored as text with 11 blank strings (all `tenure=0`, all `Churn=No`). No duplicates, no other missing values, no whitespace issues, no unexpected categories, no invalid numeric ranges.
- **Transformations:** defensive whitespace trim on string columns; `TotalCharges` → float64 with the 11 blanks set to 0.0 (factual zero billing, not imputation); `SeniorCitizen`/`tenure` normalized to int.
- **Final dataset:** 7,043 rows × 21 columns, zero missing values, unique `customerID`.
- **Decision log:** blank `TotalCharges` = 0.0 because no billing has occurred yet for tenure-0 customers (alternatives: drop rows, median imputation — rejected for reasons above).
- Raw file untouched; cleaned copy saved to `data/processed/telco_cleaned.csv`.